# Modelado experimental

## Criterios de evaluación y selección de modelos

En este proyecto, se asumirá una relación de importancia entre las clases del 70 % para la clase positiva (suscripción) y del 30 % para la clase negativa (no suscripción). Se considera que el costo asociado a un falso negativo (FN), es decir, no identificar a un cliente que se suscribiría, es superior al de un falso positivo (FP), que consiste en predecir una suscripción cuando esta no se produce. El objetivo es que el modelo sea eficaz tanto en la identificación y segmentación de estos clientes. 

1. Se priorizará el Recall como métrica principal de evaluación, con el objetivo de maximizar especialmente la proporción de suscripciones potenciales correctamente identificadas y reducir la cantidad de falsos negativos.

2. Como métrica complementaria, se utilizará el área bajo la curva Precision-Recall (PR-AUC), que permitirá evaluar la relación entre Precision y Recall a distintos umbrales de clasificación, considerando el impacto de los falsos positivos sobre la calidad de las predicciones positivas.

3. Desde una perspectiva de negocio, se analizará el costo asociado a cada modelo utilizando el umbral de clasificación predeterminado de 0.5. Este análisis permitirá comparar los modelos no solo por su capacidad predictiva, sino también por su adecuación a los costos del problema. Al mismo tiempo que se priorizara modelos interpretables, para poder tener una comprensión diagnostica inicial de posibles causas asociadas a este segmento de los usuarios.


## Librerías

In [ ]:
import os
import sys

# Agrega el directorio padre (raíz del proyecto) al path de Python
sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
import pprint

In [ ]:
# ============================================================
# 1. MANIPULACIÓN Y ANÁLISIS DE DATOS
# ============================================================

import numpy as np
import pandas as pd
import scipy.stats as stats

In [ ]:
# ============================================================
# 2. VISUALIZACIÓN DE DATOS
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import graphviz
from IPython.display import Image

In [ ]:
# ============================================================
# 3. PREPROCESAMIENTO Y REDUCCIÓN DE DIMENSIONALIDAD
# ============================================================

from sklearn.preprocessing import StandardScaler

from sklearn.decomposition import KernelPCA, PCA
from sklearn.manifold import trustworthiness

from sklearn.feature_selection import SelectKBest, f_classif, chi2
from scipy.stats import chi2_contingency

In [ ]:
# ============================================================
# 4. MODELOS DE MACHINE LEARNING
# ============================================================
from sklearn.linear_model import LinearRegression, SGDRegressor, ElasticNet

from sklearn.ensemble import (
    AdaBoostRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
    VotingRegressor,
    
)

from prophet import Prophet

from sklearn.metrics import mean_absolute_error, r2_score

from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR

from sklearn.tree import (
    DecisionTreeRegressor,
    plot_tree,
    export_text,
    export_graphviz,
    _tree
)

from xgboost import XGBRegressor

In [ ]:
# ============================================================
# 5. SELECCIÓN, VALIDACIÓN Y AJUSTE DE MODELOS
# ============================================================

from sklearn.base import clone

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    train_test_split,
)

from src.cronological_split import chronological_panel_train_test_split, ChronologicalPanelSplitCV


from sklearn.pipeline import Pipeline

from src.experiment_factory import ExperimentFactory
from src.model_optimizer import ModelOptimizer
from src.experiment_evaluator import ExperimentEvaluator
from src.experiment_tracker import ExperimentTracker

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.preprocessing import FunctionTransformer

In [ ]:
# ============================================================
# 9. FUNCIONES Y TRANSFORMACIONES PROPIAS
# ============================================================

from src.custom_standarization_classes_and_functions import GeneralPreprocessor

In [ ]:
import joblib

In [ ]:
# Función RMSPE
def rmspe(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    return np.sqrt(
        np.mean(
            ((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2
        )
    )

In [ ]:
# Define funciones nombradas (globales)
def get_week_of_year(x):
    return x.dt.isocalendar().week.astype(int)

def get_quarter(x):
    return x.dt.quarter

def get_is_weekend(x):
    return (x.dt.dayofweek >= 5).astype('category')

def get_month(x):
    return x.dt.month.astype('category')

In [ ]:
def convert_to_array(X):
    return np.asarray(X)

### Importación de datos

In [ ]:
df_ml_path = '../data/1_bronze/df.csv'
df_ml = pd.read_csv(df_ml_path)
df_ml.sample(5)

### Separación de datos

In [ ]:
#top_columns = joblib.load('../artifacts/top_columns.joblib')
#top_columns

#### `X` e `y` y `Train`-`test`

In [ ]:
X_train, X_test, y_train, y_test = chronological_panel_train_test_split(
    df_ml, 
    target_col= 'Sales',
    test_days=None, 
    test_size=0.3, 
    filter_open_train=False
)

In [ ]:
SAMPLE_SIZE = min(50000, len(X_train))
X_train = X_train.sample(n=SAMPLE_SIZE, random_state=42)
y_train = y_train.loc[X_train.index]

### Instanciación y creación de la grilla de hiperparámetros mediante GridSearchCV

En el análisis bivariado no se observaron relaciones lineales claras entre las características y la variable objetivo. Sin embargo, antes de descartar la existencia de relaciones lineales, que puedan manifestarse más allá de las asociaciones bivariadas, se explorarán distintas estrategias de reducción de dimensionalidad y modelado (incluyendo modelos de clasificación lineales y no lineales robustos, y/o con mecanismos de regularización o control de la varianza de las predicciones), teniendo en cuenta el tamaño relativamente reducido de la muestra. 

#### Estrategias de reducción de dimensionalidad

LDA (Linear Discriminant Analysis).

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
PCA = PCA()

In [ ]:
PCA_grid = {
    'n_components': [2, 4, 7],
}

Combinaciones de estrategias de reducción de dimensionalidad y grillas de hiperparámetros:

In [ ]:
reductions = {
    "options": {
        "Sin reducción": "passthrough",
        "PCA": PCA,
    },

    "grids": {
        "Sin reducción": {},
        "PCA": PCA_grid,
    },

}

#### Modelos base

* ElasticNet
* Árbol de decisión individual

ElasticNet

In [ ]:
elasticnet = ElasticNet(
    max_iter=3000,
    random_state=42
)

grid_elasticnet = {
    "alpha": [0.001, 0.01, 0.1, 1, 10, 100],
    "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
}

Arbol individual (con inyección de coste hipotetico de 0.4 para no subscritos y 0.6 para predecir subscriptos.)

In [ ]:
decision_tree = DecisionTreeRegressor(
    random_state=42
)

grid_decision_tree = {
    # Control de profundidad
    "max_depth": [3, 5, 7, 9],

    # Fracción mínima de muestras para dividir un nodo
    "min_samples_split": [0.007, 0.01, 0.02, 0.05, 0.10],

    # Fracción mínima de muestras por hoja
    "min_samples_leaf": [0.01, 0.02, 0.05, 0.10],

    # Poda del árbol
    "ccp_alpha": [0.0, 0.001, 0.01],

    # Reducción mínima de impureza para dividir
    "min_impurity_decrease": [0.0005, 0.001, 0.005, 0.01, 0.015],

    # Criterios de división
    "criterion": ["squared_error", "friedman_mse", "absolute_error"]
}

#### Modelos avanzados: ensamblados y redes neuronales

* Random Forest.
* XGBoost.
* MLPClassifier.

Random Forest

In [ ]:
random_forest = RandomForestRegressor(
    random_state=42,
    n_jobs=1
)
grid_random_forest = {
    "n_estimators": [200, 500],
    "max_depth": [None, 10, 7],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 0.5]
}

XGBoost

In [ ]:
xgboost = XGBRegressor(
    random_state=42,
    eval_metric="rmse",
    objective="reg:squarederror",
    n_jobs=1
)
grid_xgboost = {
    "n_estimators": [200, 500],
    "max_depth": [3, 6, 10],
    "learning_rate": [0.03, 0.1],
    "subsample": [0.8, 1.0]
}

MLPClassifier

In [ ]:
mlp = MLPRegressor(
    max_iter=500,
    early_stopping=True,
    random_state=42
)
grid_mlp = {
    "hidden_layer_sizes": [
        (50,),
        (100,),
        (100, 50)
    ],
    "activation": ["relu", "tanh"],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate_init": [0.001, 0.01]
}

Combinaciones de modelos y grillas de hiperparámetros:

In [ ]:
models = {
    "options": {
        #"ElasticNet": elasticnet,
        "DTR": decision_tree,
        #"RFR": random_forest,
        #"XGBR": xgboost,
        #"MLP": mlp
    },

    "grids": {
        #"ElasticNet": grid_elasticnet,
        "DTR": grid_decision_tree,
        #"RFR": grid_random_forest,
        #"XGBR": grid_xgboost,
        #"MLP": grid_mlp
    },
}

Carga de artefactos:

In [ ]:
preprocessor = joblib.load('../artifacts/pipeline_preprocessing_fe.joblib')

Combinaciones de experimentos: estrategias de reducción de dimensionalidad × estrategias de modelado × hiperparámetros.

In [ ]:
factory = ExperimentFactory(
    models=models,
    reductions=reductions,
)

experiments = factory.generate()

for experiment in experiments:
    print(f"Modelo: {experiment['nombre_modelo']}")
    print(f"Reducción: {experiment['reduccion']}")
    print(f"Grid: {experiment['param_grid']}")
    print("-" * 60)

### Entrenamiento y optimización con GridSearchCV

In [ ]:
cv = ChronologicalPanelSplitCV(
    n_splits=5,
    test_size=30,
    min_train_size=180,
    date_col='Date',
    gap=0,
    verbose=False
)

In [ ]:
""" optimizer = ModelOptimizer(
    pipeline=pipeline,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
    verbose=0
)

optimizer.fit(
    experiments=experiments,
    X=X_train,
    y=y_train
) """

### Evaluación

In [ ]:
def rmspe(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mask = y_true != 0

    return np.sqrt(
        np.mean(
            ((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2
        )
    )

---

## Pipelines individuales:

- DecisionTreeRegressor
- XGBoostRegressor

> Dado el volumen de datos y la dificultad para ejecutar GridSearch con tantos hiperparámetros, y conociendo que las metricas cmo r2 suelen estar entre 0.8 y 0.93, en competencias eficientes registradas en Kaggle. Se probaran dos algoritmos por separado, sin grid search: un arbol simple, y un xgb. La intención sera principalmente evaluar que tanto mejora al utilizar un modelo complejo; y si es conveniente solo conservar el arbol simple, a favor de menor costo de reentrenamiento, menor latencia, y mayor interpretabilidad.



Pipeline borrador DT:

In [ ]:
# Pipeline Decision Tree
pipeline_ml_dt = Pipeline(
    steps=[
        ('preprocessing', preprocessor),
        ('dt', DecisionTreeRegressor(
            max_depth=15,
            min_samples_split=0.05,
            min_samples_leaf=0.05,
            ccp_alpha=0.001,
            min_impurity_decrease=0.003,
            criterion="friedman_mse",
            random_state=42
        ))
    ]
)

# Entrenamiento
pipeline_ml_dt.fit(X_train, y_train)

# Predicciones
preds_train = pipeline_ml_dt.predict(X_train)
preds_test = pipeline_ml_dt.predict(X_test)

# Evaluación TRAIN
print('======= TRAIN ========')
print('MAE:', mean_absolute_error(y_train, preds_train))
print('R2:', r2_score(y_train, preds_train))
print('RMSPE:', rmspe(y_train, preds_train))

# Evaluación TEST
print('\n======= TEST ========')
print('MAE:', mean_absolute_error(y_test, preds_test))
print('R2:', r2_score(y_test, preds_test))
print('RMSPE:', rmspe(y_test, preds_test))

In [ ]:
pipeline_ml_xgb = Pipeline(
    steps=[
        ('preprocessing', preprocessor),
        ('to_array', FunctionTransformer(convert_to_array)),
        ('xgb', XGBRegressor(
            # Complejidad de los árboles

            max_depth=3,
            min_child_weight=10,

            # Aprendizaje conservador

            learning_rate=0.02,
            n_estimators=500,

            # Regularización
            reg_alpha=1.0,
            reg_lambda=10.0,
            gamma=0.5,

            # Submuestreo

            subsample=0.7,
            colsample_bytree=0.7,

            # Objetivo
            objective="reg:squarederror",
            eval_metric="rmse",

            # Reproducibilidad

            random_state=42,
            n_jobs=-1

        ))
    ]
)

# Entrenamiento
pipeline_ml_xgb.fit(X_train, y_train)

# Predicciones
preds_train = pipeline_ml_xgb.predict(X_train)
preds_test = pipeline_ml_xgb.predict(X_test)

# Evaluación TRAIN
print('======= TRAIN =======')
print('MAE:', mean_absolute_error(y_train, preds_train))
print('R2:', r2_score(y_train, preds_train))
print('RMSPE:', rmspe(y_train, preds_train))

# Evaluación TEST
print('\n======= TEST =======')
print('MAE:', mean_absolute_error(y_test, preds_test))
print('R2:', r2_score(y_test, preds_test))
print('RMSPE:', rmspe(y_test, preds_test))

#### Evaluación individual

##### Árbol de decisión simple

In [ ]:
df_experiments[(df_experiments['Modelo']=='DTC') & (df_experiments['Reducción']=='Sin reducción')].plot(kind='bar')

##### Interpretabilidad individual

In [ ]:
nombres_variables = (
    pipeline_decision_tree_sin_reduccion
    .named_steps['preprocessor']
    .get_feature_names_out()
)

In [ ]:
# Asociar las clases originales con sus etiquetas

mapa_clases = {
    0: "Usuario tiende a no suscribirse",
    1: "Usuario tiende a suscribirse"
}

# Obtener los nombres respetando el orden de las clases del modelo

nombre_clases = [mapa_clases[clase] for clase in pipeline_decision_tree_sin_reduccion.classes_]

In [ ]:
# Crear la figura
fig, ax = plt.subplots(figsize=(24, 12))

plot_tree(
    decision_tree_sin_reduccion,
    max_depth=15,  # Mostrar solo los primeros niveles
    filled=True,
    rounded=True,
    feature_names=nombres_variables,
    class_names=nombre_clases,
    fontsize=9,
    proportion=True,
    precision=2,
    impurity=False,
    ax=ax
)

ax.set_title(
    "Árbol de decisión - Vista general",
    fontsize=16,
    pad=20
)

plt.tight_layout()

plt.show()

In [ ]:
pd.DataFrame({'Columnas preprocesadas':nombres_variables, 'Importancia':decision_tree_sin_reduccion.feature_importances_}).sort_values(by='Importancia', ascending=False).round(10)#['Columnas preprocesadas']

##### Interpretabilidad individual

In [ ]:
pd.DataFrame({'Columns':preprocessor.get_feature_names_out(), 'Coeficientes':elasticnet_sin_reduccion.coef_[0]}).sort_values(by=['Coeficientes'], ascending=False)

#### Comparativa de los modelos seleccionados

In [ ]:
df_experiments

In [ ]:
df_experiments_filtrado = df_experiments[
    (df_experiments['Modelo'].isin(['DTC', 'ElasticNet'])) &
    (df_experiments['Reducción'] == 'Sin reducción')
]

metricas = ['cv mean test score', 'cv mean train score', 'cv std test score', 'Test average_precision', 'Test cost_loss']

df_largo = df_experiments_filtrado.melt(
    id_vars=['Modelo'],
    value_vars=metricas,
    var_name='Métrica',
    value_name='Valor'
)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=df_largo,
    x='Métrica',
    y='Valor',
    hue='Modelo'
)

plt.title('Comparación de métricas: DTC vs ElasticNet')
plt.xlabel('Métrica')
plt.ylabel('Valor')
plt.legend(title='Modelo')
plt.tight_layout()
plt.show()

### Guardado del mejor modelo

Modelo.

In [ ]:
# Guardar el modelo entrenado en un archivo .pkl
path_best_ml_model = '../models/subscription_status_decision_tree_base_model.pkl'
joblib.dump(decision_tree_sin_reduccion, path_best_ml_model)

Pipeline completo.

In [ ]:

# Guardar el modelo entrenado en un archivo .pkl
path_best_ml_model_pipeline = '../models/pipeline_decision_tree_sin_reduccion.pkl'
joblib.dump(pipeline_decision_tree_sin_reduccion, path_best_ml_model_pipeline)

---